
Building a GAN-Based AI Text Detector

Last Updated: July 14th, 2025

Daily Challenge : Building a GAN-Based AI Text Detector


👩‍🏫 👩🏿‍🏫 What You’ll learn

    How to train a Generative Adversarial Network (GAN) for detecting AI-generated text.
    How to use a pre-trained BERT model for sequence classification.
    How to preprocess text data and tokenize it for deep learning models.
    How to evaluate model performance using AUC scores.
    How to fine-tune and optimize deep learning models.
    How to perform inference and generate predictions on test data.


🛠️ What you will create

    A GAN-based model that detects AI-generated text using embeddings from a BERT model.
    A training pipeline that leverages a discriminator and generator network.
    A model that improves based on AUC scores for stability in training.
    A final submission file with predictions on the test dataset.


Dataset

You can find the dataset for this exercise here: Dataset


Task

For today’s challenge, you are provided with the final code with parts to fill. When you see a “TODO” it means you need to write code. Complete all of them.

Instructions :

1. Download the Dataset

    Upload the Kaggle API key.
    Move the key to the correct directory and set permissions, you may accept the rules of the competitions in Rulesor in Participate.
    Download and unzip the dataset.
    or :
    Download manually from Kaggle

2. Load the Data

    Read the training and test datasets using pandas.
    Display basic statistics and structure of the dataset.

3. Prepare the Model

    Load the BERT tokenizer and pre-trained model for sequence classification : bert-base-uncased.
    Extract embeddings from the BERT model to use in the GAN framework.

4. Set Hyperparameters

    Define batch sizes, learning rates, latent vector dimensions, and training epochs.

5. Prepare the Data for Training

    Create a PyTorch dataset class for handling text data.
    Split the data into training and testing sets.
    Use DataLoader to load batches efficiently.

6. Define the Generator Model

    Build a neural network that generates text embeddings using ConvTranspose1D layers.
    Incorporate a BERT encoder in the generator.

7. Define the Discriminator Model

    Extract and modify layers from a pre-trained BERT model.
    Implement a pooling mechanism for text classification.
    Construct a classification head using fully connected layers.

8. Train the Model

    Implement a GAN training loop.
    Train the generator to produce embeddings that fool the discriminator.
    Train the discriminator to differentiate between real and generated embeddings.
    Evaluate the model using AUC scores to monitor training stability.

9. Perform Inference

    Load the best-performing discriminator model based on AUC scores.
    Process test data through the model to generate predictions.




In [2]:
import pandas as pd
import csv

TRAIN_PATH = "train_essays.csv"  # adapte si besoin

# 1) Lecture robuste
df = pd.read_csv(TRAIN_PATH, engine="python")  # engine="python" pour tolérer certains cas

print("Colonnes (repr):", list(map(repr, df.columns)))
print(df.head(2).to_string())

# 2) Strip des noms de colonnes pour virer espaces/retours à la ligne
df.columns = df.columns.str.strip()

# 3) Vérifier que 'generated' est bien là
if 'generated' not in df.columns:
    # Peut-être 'Generated', 'label', 'is_generated', etc.
    print("Colonne 'generated' introuvable. Colonnes présentes :", df.columns.tolist())
else:
    print("\nType de df['generated']:", df['generated'].dtype)
    print("Nb de NaN dans 'generated':", df['generated'].isna().sum())
    print("Valeurs uniques (brutes):", df['generated'].unique()[:10])

    # 4) Forcer en numérique si c'est du string "0"/"1"
    df['generated'] = pd.to_numeric(df['generated'], errors='coerce')
    print("Après conversion -> NaN:", df['generated'].isna().sum())

    # 5) Stat rapide
    print(df['generated'].value_counts(dropna=False, normalize=True).head())

# 6) Sanity check sur le test
TEST_PATH = "test_essays.csv"
test_df = pd.read_csv(TEST_PATH)
print("\nTest columns:", test_df.columns.tolist())

Colonnes (repr): ["'id'", "'prompt_id'", "'text'", "'generated'"]
         id  prompt_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

il n’y a pas de bug de lecture. La colonne generated existe, est bien typée en int64, sans NaN, et contient des 0 et des 1. L’impression “qu’elle est vide” vient du déséquilibre massif :

Split validation à stratifier absolument
Sinon tu peux tomber sur un fold/val avec ~0 ou 1 seul positif → AUC instable / inutilisable.

Perte pondérée ou sampler

    Utiliser BCEWithLogitsLoss(pos_weight = n_neg / n_pos) (si tu gardes la version “logits” du discriminateur).

    Ou un WeightedRandomSampler pour rééquilibrer les batchs (attention au sur-apprentissage des rares positifs).

    Ou pré-entraîner le discriminateur en supervision pure (classification binaire) avec pondération, puis lancer la boucle GAN (option souvent plus stable).

Suivre aussi l’AUC-PR (average precision)
Avec un dataset aussi déséquilibré, l’AUC-PR est souvent plus informative que l’AUC-ROC.

Early stopping + seuil optimisé
Garde le meilleur modèle au max AUC (ou AUC-PR) et optimise ton seuil de décision sur la val (Youden J, F1-max, etc.).

In [ ]:
import os, random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# ===== Kaggle (uploade ton kaggle.json dans /content d'abord) =====
!mkdir -p /root/.kaggle
!cp /content/kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# ===== Téléchargement et unzip =====
!kaggle competitions download -c llm-detect-ai-generated-text -p /content/data
!unzip -o /content/data/llm-detect-ai-generated-text.zip -d /content/data

NameError: name 'src_train' is not defined

In [5]:
import os
import numpy as np
import pandas as pd
from typing import Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import BertConfig
from transformers.models.bert.modeling_bert import BertEncoder
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedShuffleSplit

In [7]:
# ----------------------------------
# Device
# ----------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ----------------------------------
# Paths
# ----------------------------------
DATA_DIR = ""

TRAIN_PATH  = os.path.join(DATA_DIR, "train_essays.csv")
TEST_PATH   = os.path.join(DATA_DIR, "test_essays.csv")
PROMPT_PATH = os.path.join(DATA_DIR, "train_prompts.csv")
SUB_PATH    = os.path.join(DATA_DIR, "sample_submission.csv")

src_train  = pd.read_csv(TRAIN_PATH)
src_prompt = pd.read_csv(PROMPT_PATH)
src_sub    = pd.read_csv(SUB_PATH)

# Nettoyage des noms de colonnes (par précaution)
src_train.columns = src_train.columns.str.strip()

assert "text" in src_train.columns
assert "generated" in src_train.columns

print("Distribution des labels (train):")
print(src_train["generated"].value_counts(normalize=True))

# ----------------------------------
# Model preparation
# ----------------------------------
tokenizer_save_path = "./tokenizer"
model_save_path = "./best_discriminator.pt"

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# Discriminateur final binaire → num_labels=1 (mais on n'utilise pas la tête de classification)
pretrained_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=1)
embedding_model = pretrained_model.bert.to(device)
embedding_model.eval()

# ----------------------------------
# Hyperparameters
# ----------------------------------
train_batch_size   = 32
val_batch_size     = 64
infer_batch_size   = 64

lr_D = 2e-4
lr_G = 2e-4
beta1 = 0.5

nz = 100                 # Dimensions du vecteur latent
num_epochs_gan = 3       # Époques de GAN (peut être augmenté)
num_epochs_sup = 2       # Époques de pré-entraînement supervisé du D

num_hidden_layers = 6    # Nombre de couches qu'on garde dans le D (sur les 12 de BERT)
train_ratio = 0.8        # split train/val
max_seq_len = 128

# ----------------------------------
# Data Preparation
# ----------------------------------
class GANDAIGDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels  # peut être None pour test

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        if self.labels is None:
            return self.texts[idx]
        return self.texts[idx], self.labels[idx]

# Split STRATIFIÉ (important vu le fort déséquilibre)
y = src_train["generated"].values
X = src_train["text"].values

sss = StratifiedShuffleSplit(n_splits=1, test_size=1 - train_ratio, random_state=42)
train_idx, val_idx = next(sss.split(X, y))

train_set = src_train.iloc[train_idx].reset_index(drop=True)
val_set   = src_train.iloc[val_idx].reset_index(drop=True)
test_set  = pd.read_csv(TEST_PATH)

# pos_weight pour BCEWithLogitsLoss
n_pos = (train_set["generated"] == 1).sum()
n_neg = (train_set["generated"] == 0).sum()
pos_weight_tensor = torch.tensor([n_neg / max(n_pos, 1)], device=device)
print("pos_weight:", pos_weight_tensor.item(), " (#pos:", n_pos, ", #neg:", n_neg, ")")

train_dataset = GANDAIGDataset(train_set["text"].tolist(), train_set["generated"].tolist())
val_dataset   = GANDAIGDataset(val_set["text"].tolist(),   val_set["generated"].tolist())

# (Optionnel) Sampler pondéré — ici on reste sur pos_weight dans la loss
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_dataset,   batch_size=val_batch_size,   shuffle=False, drop_last=False)

# ----------------------------------
# Models
# ----------------------------------
config = BertConfig(
    hidden_size=768,
    num_hidden_layers=num_hidden_layers,
    num_attention_heads=12,
    intermediate_size=3072,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)

class Generator(nn.Module):
    """
    z -> (batch, seq_len, hidden_size)
    """
    def __init__(self, input_dim, seq_len=128, hidden_size=768):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_size = hidden_size

        self.fc = nn.Linear(input_dim, 256 * seq_len)

        self.conv_net = nn.Sequential(
            nn.ConvTranspose1d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),

            nn.ConvTranspose1d(512, hidden_size, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(inplace=True),
        )

        self.bert_encoder = BertEncoder(config)

    def forward(self, x):
        # x: (batch, nz)
        bsz = x.size(0)
        x = self.fc(x)                               # (batch, 256*seq_len)
        x = x.view(bsz, 256, self.seq_len)           # (batch, 256, seq_len)
        x = self.conv_net(x)                         # (batch, 768, seq_len)
        x = x.permute(0, 2, 1).contiguous()          # (batch, seq_len, 768)

        attn_mask = torch.ones(bsz, 1, 1, self.seq_len, device=x.device)
        out = self.bert_encoder(
            hidden_states=x,
            attention_mask=attn_mask,
            return_dict=True
        )
        return out  # .last_hidden_state

class SumBertPooler(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # hidden_states: (batch, seq_len, hidden_size)
        sum_hidden = hidden_states.sum(dim=1)        # (batch, hidden_size)
        sum_mask = sum_hidden.sum(1).unsqueeze(1)    # (batch, 1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_hidden / sum_mask
        return mean_embeddings

class Discriminator(nn.Module):
    """
    Renvoie des **logits** (pas de sigmoid dans forward).
    """
    def __init__(self):
        super().__init__()
        self.bert_encoder = BertEncoder(config)
        self.bert_encoder.layer = nn.ModuleList([
            layer for layer in pretrained_model.bert.encoder.layer[:num_hidden_layers]
        ])
        self.pooler = SumBertPooler()
        self.classifier = torch.nn.Sequential(
            nn.Linear(config.hidden_size, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )

    def forward(self, input):
        bsz, seq_len, _ = input.size()
        attn_mask = torch.ones(bsz, 1, 1, seq_len, device=input.device)
        out = self.bert_encoder(
            hidden_states=input,
            attention_mask=attn_mask,
            return_dict=True
        )
        out = self.pooler(out.last_hidden_state)  # (batch, hidden)
        out = self.classifier(out)                # (batch, 1) logits
        return out.view(-1)

# ----------------------------------
# Utils
# ----------------------------------
@torch.no_grad()
def get_embeddings(texts: list) -> torch.Tensor:
    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_seq_len,
        return_tensors="pt"
    ).to(device)

    outputs = embedding_model(
        input_ids=encodings["input_ids"],
        token_type_ids=encodings["token_type_ids"],
        attention_mask=encodings["attention_mask"]
    )
    return outputs.last_hidden_state  # (batch, seq_len, hidden)

@torch.no_grad()
def evaluate(model: nn.Module, data_loader: DataLoader) -> Tuple[float, float]:
    model.eval()
    preds = []
    trues = []
    for texts, labels in data_loader:
        emb = get_embeddings(texts)
        logits = model(emb)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds.extend(probs)
        trues.extend(labels.numpy())
    auc = roc_auc_score(trues, preds)
    ap  = average_precision_score(trues, preds)
    return auc, ap

def get_model_info_dict(model, epoch, auc_score, ap_score):
    current_device = next(model.parameters()).device
    model.to('cpu')
    model_info = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'auc_score': auc_score,
        'ap_score': ap_score
    }
    model.to(current_device)
    return model_info

# ----------------------------------
# 2) Pré-entraînement supervisé du Discriminateur
# ----------------------------------
netD = Discriminator().to(device)
criterion_sup = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizerD_sup = optim.Adam(netD.parameters(), lr=lr_D, betas=(beta1, 0.999))

print("==> Pré-entraînement supervisé du Discriminateur ...")
best_sup = {"epoch": -1, "auc": -1.0, "state": None}
for epoch in range(num_epochs_sup):
    netD.train()
    running_loss = 0.0
    for texts, labels in train_loader:
        labels = labels.float().to(device)
        emb = get_embeddings(texts)

        logits = netD(emb)
        loss = criterion_sup(logits, labels)
        optimizerD_sup.zero_grad()
        loss.backward()
        optimizerD_sup.step()
        running_loss += loss.item() * labels.size(0)

    auc, ap = evaluate(netD, val_loader)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"[Sup D] Epoch {epoch+1}/{num_epochs_sup} - loss: {epoch_loss:.4f} | AUC: {auc:.4f} | AUPRC: {ap:.4f}")
    if auc > best_sup["auc"]:
        best_sup = {"epoch": epoch, "auc": auc, "state": netD.state_dict()}

# Charger le meilleur D supervisé pour démarrer le GAN
netD.load_state_dict(best_sup["state"])
print(f"Loaded best supervised D (epoch={best_sup['epoch']}, AUC={best_sup['auc']:.4f})")

# ----------------------------------
# 3) Entraînement GAN
# ----------------------------------
netG = Generator(input_dim=nz, seq_len=max_seq_len, hidden_size=768).to(device)
criterion_gan = nn.BCEWithLogitsLoss()  # Ici, pas de pos_weight : GAN "pur"
optimizerD = optim.Adam(netD.parameters(), lr=lr_D, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr_G, betas=(beta1, 0.999))

def GAN_step(optimizerG, optimizerD, netG, netD, real_data, epoch, i):
    netD.train(); netG.train()

    bsz = real_data.size(0)
    # Labels GAN: vrais=1, faux=0
    real_targets = torch.ones(bsz, device=device)
    fake_targets = torch.zeros(bsz, device=device)

    # ---- D step ----
    optimizerD.zero_grad()
    logits_real = netD(real_data)
    loss_real = criterion_gan(logits_real, real_targets)

    noise = torch.randn(bsz, nz, device=device)
    fake_data = netG(noise).last_hidden_state
    logits_fake = netD(fake_data.detach())
    loss_fake = criterion_gan(logits_fake, fake_targets)

    loss_D = loss_real + loss_fake
    loss_D.backward()
    optimizerD.step()

    # ---- G step ----
    optimizerG.zero_grad()
    logits_fake_for_G = netD(fake_data)  # le G veut faire croire au D que c'est vrai => target=1
    loss_G = criterion_gan(logits_fake_for_G, real_targets)
    loss_G.backward()
    optimizerG.step()

    if i % 50 == 0:
        with torch.no_grad():
            D_x = torch.sigmoid(logits_real).mean().item()
            D_G_z1 = torch.sigmoid(logits_fake).mean().item()
            D_G_z2 = torch.sigmoid(logits_fake_for_G).mean().item()
        print('[GAN][%d/%d][%d/%d] Loss_D: %.4f Loss_G: %.4f D(x): %.4f D(G(z)): %.4f / %.4f'
              % (epoch, num_epochs_gan, i, len(train_loader), loss_D.item(), loss_G.item(), D_x, D_G_z1, D_G_z2))

model_infos = []
print("==> Entraînement GAN ...")
for epoch in range(num_epochs_gan):
    for i, (texts, labels) in enumerate(train_loader):
        with torch.no_grad():
            real_embeddings = get_embeddings(texts)
        GAN_step(
            optimizerG=optimizerG,
            optimizerD=optimizerD,
            netG=netG,
            netD=netD,
            real_data=real_embeddings,
            epoch=epoch,
            i=i
        )

    auc, ap = evaluate(netD, val_loader)
    model_infos.append(get_model_info_dict(netD, epoch, auc, ap))
    print(f"[GAN] Epoch {epoch+1}/{num_epochs_gan} - AUC: {auc:.4f} | AUPRC: {ap:.4f}")

print('Train complete！')

# ----------------------------------
# 4) Inference (meilleur modèle selon AUC)
# ----------------------------------
max_auc_model_info = max(model_infos, key=lambda x: x['auc_score'])
torch.save(max_auc_model_info, model_save_path)
print("Best GAN epoch:", max_auc_model_info["epoch"], "AUC:", max_auc_model_info["auc_score"], "AUPRC:", max_auc_model_info["ap_score"])

model = Discriminator().to(device)
model.load_state_dict(max_auc_model_info['model_state_dict'])
model.eval()

class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, texts):
        self.texts = texts
    def __getitem__(self, idx):
        return self.texts[idx]
    def __len__(self):
        return len(self.texts)

sub_dataset = InferenceDataset(test_set["text"].tolist())
inference_loader = DataLoader(sub_dataset, batch_size=infer_batch_size, shuffle=False, drop_last=False)

sub_predictions = []
with torch.no_grad():
    for texts in inference_loader:
        emb = get_embeddings(texts)
        logits = model(emb)
        probs = torch.sigmoid(logits).cpu().numpy()
        sub_predictions.extend(probs)

sub_ans_df = pd.DataFrame({
    "id": test_set["id"],
    "generated": sub_predictions
})
print(sub_ans_df.head())

sub_path = "/content/submission.csv"
sub_ans_df.to_csv(sub_path, index=False)
print("Saved submission to:", sub_path)

Device: cuda
Distribution des labels (train):
generated
0    0.997823
1    0.002177
Name: proportion, dtype: float64


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\mathi\Downloads\GenAI\GenAI_Bootcamp\nlp_env\lib\site-packages\torch\cuda\__init__.py:218: UserWarning: 
NVIDIA GeForce RTX 5060 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5060 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


pos_weight: 550.0  (#pos: 2 , #neg: 1100 )
==> Pré-entraînement supervisé du Discriminateur ...


RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:

device = torch.device("cpu")

TRAIN_PATH = "train_essays.csv"
TEST_PATH = "test_essays.csv"
PROMPT_PATH = "train_prompts.csv"
SUB_PATH = "sample_submission.csv"

src_train = pd.read_csv(TRAIN_PATH)
src_test = pd.read_csv(TEST_PATH)
src_prompt = pd.read_csv(PROMPT_PATH)
src_sub = pd.read_csv(SUB_PATH)

# Model preparation

tokenizer_save_path = "bert_tokenizer"
model_save_path = "bert_model"

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
pretrained_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=1)
embedding_model = BertModel.from_pretrained("bert-base-uncased")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
train_batch_size = 16
test_batch_size = 32
lr = 2e-5
beta1 = 0.5
num_epochs = 3
num_hidden_layers = 4
train_ratio = 0.9

In [5]:
print(src_train.columns)

Index(['id', 'prompt_id', 'text', 'generated'], dtype='object')


In [6]:


all_num = len(src_train)
train_num = int(all_num * train_ratio)

texts = src_train["text"].tolist()
labels = src_train["generated"].tolist()

X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=1-train_ratio, random_state=42)

train_set = pd.DataFrame({"text": X_train, "generated": y_train})
val_set = pd.DataFrame({"text": X_val, "generated": y_val})
test_set = src_test  # no labels

In [16]:
train_batch_size = 16
test_batch_size = 32
lr = 2e-5
beta1 = 0.5
num_epochs = 3
num_hidden_layers = 4
train_ratio = 0.9
nz = 100  # Dimensions of the latent vector

In [8]:

class GANDAIGDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


In [9]:
all_num = len(src_train)
train_num = int(all_num * train_ratio)
test_num = all_num - train_num


train_set = pd.concat([
    src_train.iloc[:train_num].reset_index(drop=True),
    src_prompt.iloc[:train_num].reset_index(drop=True)
]).reset_index(drop=True)
val_set = pd.concat([
    src_train.iloc[train_num:].reset_index(drop=True),
    src_prompt.iloc[train_num:].reset_index(drop=True)
]).reset_index(drop=True)
test_set = pd.concat([
    src_test,
    src_prompt.iloc[train_num:].reset_index(drop=True)
]).reset_index(drop=True)

train_dataset = GANDAIGDataset(train_set["text"].tolist(), train_set["generated"].tolist())
test_dataset = GANDAIGDataset(val_set["text"].tolist(), val_set["generated"].tolist())

train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)


In [10]:
config = BertConfig(num_hidden_layers=num_hidden_layers)
class Generator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Linear(input_dim, 256 * 128)

        self.conv_net = nn.Sequential(
            nn.ConvTranspose1d(256, 128, kernel_size=4, stride=2, padding=1),  # (128, 256)
            nn.BatchNorm1d(128),
            nn.ReLU(True),

            nn.ConvTranspose1d(128, 64, kernel_size=4, stride=2, padding=1),   # (64, 512)
            nn.BatchNorm1d(64),
            nn.ReLU(True),

            nn.ConvTranspose1d(64, 32, kernel_size=4, stride=2, padding=1),    # (32, 1024)
            nn.BatchNorm1d(32),
            nn.ReLU(True),

            nn.ConvTranspose1d(32, 768, kernel_size=3, stride=1, padding=1),   # (768, 1024)
        )

        self.bert_encoder = BertEncoder(config)

    def forward(self, x):
        x = self.fc(x)  # shape: (batch_size, 256 * 128)
        x = x.view(-1, 256, 128)  # reshape pour ConvTranspose1D
        x = self.conv_net(x)  # (batch_size, 768, seq_len)

        x = x.permute(0, 2, 1)  # (batch_size, seq_len, 768) pour le BERT encoder
        attention_mask = torch.ones(x.shape[:2], dtype=torch.long, device=x.device)
        extended_attention_mask = attention_mask[:, None, None, :]
        extended_attention_mask = (1.0 - extended_attention_mask) * -10000.0

        encoder_output = self.bert_encoder(x, attention_mask=extended_attention_mask)
        return encoder_output

In [26]:
class SumBertPooler(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        sum_hidden = hidden_states.sum(dim=1)
        sum_mask = sum_hidden.sum(1).unsqueeze(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)

        mean_embeddings = sum_hidden / sum_mask
        return mean_embeddings

In [27]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert_encoder = BertEncoder(config)
        self.bert_encoder.layer = nn.ModuleList([
            layer for layer in pretrained_model.bert.encoder.layer[:6]
        ])
        self.pooler = SumBertPooler()
        self.classifier = torch.nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )

    def forward(self, input):
        out = self.bert_encoder(input)
        out = self.pooler(out.last_hidden_state)
        out = self.classifier(out)
        return torch.sigmoid(out).view(-1)

In [28]:
def eval_auc(model):
    model.eval()

    predictions = []
    actuals = []
    with torch.no_grad():
        for batch in test_loader:
            # Texte et labels
            texts = batch[0]
            labels = batch[1].float().to(device)

            # Encodage avec le tokenizer
            encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
            input_ids = encodings['input_ids']
            token_type_ids = encodings['token_type_ids']
            attention_mask = encodings['attention_mask']

            # Embeddings via BERT
            embeded = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids).last_hidden_state

            # Prédictions
            outputs = model(embeded, attention_mask=attention_mask)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(labels.cpu().numpy())

    # Calcul de l'AUC
    auc = roc_auc_score(actuals, predictions)
    print("AUC:", auc)
    return auc

In [29]:
def get_model_info_dict(model, epoch, auc_score):
    current_device = next(model.parameters()).device
    model.to('cpu')
    model_info = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'auc_score': auc_score,
    }
    model.to(current_device)
    return model_info

def preparation_embedding(texts):
    encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    input_ids = encodings['input_ids']
    token_type_ids = encodings['token_type_ids']
    outputs = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids)
    return outputs.last_hidden_state

def GAN_step(optimizerG, optimizerD, netG, netD, real_data, label, epoch, i):
    netD.zero_grad()
    batch_size = real_data.size(0)

    output_real = netD(real_data)
    errD_real = criterion(output_real, label)
    errD_real.backward()
    D_x = output_real.mean().item()

    noise = torch.randn(batch_size, nz, device=device)
    fake_data = netG(noise).last_hidden_state
    label.fill_(1)
    output_fake = netD(fake_data.detach())
    errD_fake = criterion(output_fake, label)
    errD_fake.backward()
    D_G_z1 = output_fake.mean().item()
    errD = errD_real + errD_fake
    optimizerD.step()

    netG.zero_grad()
    label.fill_(0)
    output_fake2 = netD(fake_data)
    errG = criterion(output_fake2, label)
    errG.backward()
    D_G_z2 = output_fake2.mean().item()
    optimizerG.step()

    if i % 50 == 0:
        print('[%d/%d][%d/%d] Loss_D: %.4f Loss_G: %.4f D(x): %.4f D(G(z)): %.4f / %.4f'
              % (epoch, num_epochs, i, len(train_loader), errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

    return optimizerG, optimizerD, netG, netD

In [30]:
netG = Generator(input_dim=nz).to(device)
netD = Discriminator().to(device)

criterion = nn.BCELoss()

optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

In [ ]:
model_infos = []

for epoch in range(num_epochs):
    for i, data in enumerate(train_loader, 0):
        with torch.no_grad():
            embeded = preparation_embedding(data[0])

        optimizerG, optimizerD, netG, netD = GAN_step(
            optimizerG=optimizerG,
            optimizerD=optimizerD,
            netG=netG,
            netD=netD,
            real_data=embeded.to(device),
            label=data[1].float().to(device),
            epoch=epoch,
            i=i
        )

    auc_score = eval_auc(netD)
    model_infos.append(get_model_info_dict(netD, epoch, auc_score))

print('Train complete！')

In [ ]:
# Sélection du meilleur modèle
max_auc_model_info = max(model_infos, key=lambda x: x['auc_score'])

model = Discriminator()
model.load_state_dict(max_auc_model_info['model_state_dict'])
model.to(device)
model.eval()

In [ ]:
class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __getitem__(self, idx):
        return self.texts[idx]

    def __len__(self):
        return len(self.texts)

sub_dataset = InferenceDataset(src_test["text"].tolist())
inference_loader = DataLoader(sub_dataset, batch_size=test_batch_size, shuffle=False)

sub_predictions = []

with torch.no_grad():
    for batch in inference_loader:
        encodings = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
        input_ids = encodings['input_ids']
        token_type_ids = encodings['token_type_ids']
        embeded = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids).last_hidden_state
        embeded = embeded.to(device)

        outputs = model(embeded)
        sub_predictions.extend(outputs.cpu().numpy())

In [ ]:
sub_ans_df = pd.DataFrame({
    "id": src_test["id"],
    "generated": sub_predictions
})

print(sub_ans_df.head())

# Sauvegarde finale
sub_ans_df.to_csv("submission.csv", index=False)